# YOLOv8 Fine-Tuning for APCOMS Shuttle Passenger Detection

Fine-tunes the base YOLOv8n model on shuttle entrance footage so person
detection is tuned to our deployment camera angle, lighting conditions,
and demographic.

**Dataset:** ~300 labeled frames from simulated Makerere shuttle entrance footage with diverse people.

**Output:** A `best.pt` file that replaces `counting_module/models/yolov8n.pt`
for improved detection accuracy in production.

---

## 0. Environment setup

In [ ]:
# Mount Google Drive so we can save weights persistently across sessions.
# When prompted, sign in with the SAME Gmail used to create APCOMS_YOLO_runs.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install ultralytics (provides the YOLO class and CLI). Pinned to a
# stable version so future runs reproduce today's training behaviour.
!pip install -q ultralytics==8.3.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-pyth

In [ ]:
# Confirm GPU availability. Should print T4, A100, or similar.
# If this prints "command not found" or no GPU, go to
# Runtime > Change runtime type > Hardware accelerator > T4 GPU.
!nvidia-smi

Mon Jun 15 06:06:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

---

## 1. Dataset download

The labeled dataset is exported from Label Studio (local install on
the development laptop) as a YOLO-format zip, then uploaded to the
APCOMS_YOLO_runs folder in Google Drive. This keeps training data
under our control -- it never sits on a third-party server.

Expected Drive structure after upload:
APCOMS_YOLO_runs/
  shuttle_dataset.zip   <- exported from Label Studio

In [ ]:
# Working directory for this session. Colab's local filesystem is
# wiped when the session ends; we save final artifacts to Drive.
import os
os.chdir('/content')

# Copy the labeled dataset zip from Drive into Colab's local
# filesystem, then unzip. The zip should already be uploaded to
# APCOMS_YOLO_runs/ in Drive before running this notebook.
DATASET_ZIP = "/content/drive/MyDrive/APCOMS_YOLO_runs/shuttle_dataset.zip"

!cp "$DATASET_ZIP" /content/dataset.zip
!unzip -q /content/dataset.zip -d /content/dataset
!ls /content/dataset

replace /content/dataset/classes.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
classes.txt  data.yaml	images	labels	notes.json  test  train  val


In [ ]:
# Restructure Label Studio's flat YOLO export into the train/val/test
# layout YOLOv8 expects. Splits 70/20/10 with deterministic shuffling
# so re-runs produce the same split (reproducibility for the panel).
import os
import random
import shutil
from pathlib import Path

DATASET_ROOT = Path("/content/dataset")
IMAGES_DIR = DATASET_ROOT / "images"
LABELS_DIR = DATASET_ROOT / "labels"

# Collect every image and pair it with its matching label file.
image_files = sorted(IMAGES_DIR.glob("*.jpg")) + sorted(IMAGES_DIR.glob("*.png"))
print(f"Found {len(image_files)} images in the flat export.")

# Deterministic shuffle — seeded so the split is reproducible.
random.seed(42)
random.shuffle(image_files)

# 70 / 20 / 10 split
total = len(image_files)
train_end = int(total * 0.70)
val_end = train_end + int(total * 0.20)

splits = {
    "train": image_files[:train_end],
    "val": image_files[train_end:val_end],
    "test": image_files[val_end:],
}

print(f"Train: {len(splits['train'])}  Val: {len(splits['val'])}  Test: {len(splits['test'])}")

# Create the YOLO directory layout and move files into it.
for split_name, files in splits.items():
    img_out = DATASET_ROOT / split_name / "images"
    lbl_out = DATASET_ROOT / split_name / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for img_path in files:
        # Move image
        shutil.move(str(img_path), str(img_out / img_path.name))
        # Move matching label file (same stem, .txt extension)
        label_path = LABELS_DIR / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.move(str(label_path), str(lbl_out / label_path.name))

# Remove the now-empty original flat folders.
shutil.rmtree(IMAGES_DIR, ignore_errors=True)
shutil.rmtree(LABELS_DIR, ignore_errors=True)

# Write data.yaml so YOLO knows where everything is.
data_yaml = f"""path: {DATASET_ROOT}
train: train/images
val: val/images
test: test/images

nc: 1
names: ['person']
"""

(DATASET_ROOT / "data.yaml").write_text(data_yaml)
print("\nWrote data.yaml:")
print(data_yaml)

# Final sanity check
!ls -R /content/dataset | head -20

Found 327 images in the flat export.
Train: 228  Val: 65  Test: 34

Wrote data.yaml:
path: /content/dataset
train: train/images
val: val/images
test: test/images

nc: 1
names: ['person']

/content/dataset:
classes.txt
data.yaml
notes.json
test
train
val

/content/dataset/test:
images
labels

/content/dataset/test/images:
06173c94-VID_20260612_1736200882_frame_0023.jpg
092e611f-VID_20260612_202345203_frame_0104.jpg
093d5fd9-VID_20260526_132322811_frame_0008.jpg
09a9626f-VID-20260612-WA0002_frame_0019.jpg
09c41ea9-VID_20260612_202345203_02_frame_0077.jpg
1bc30feb-VID_20260526_1322586452_frame_0005.jpg
210f679a-VID_20260612_1734174082_frame_0005.jpg


In [ ]:
# Verify dataset structure. data.yaml is the file YOLO reads to know
# class names and folder locations. Print it so we can spot issues
# (wrong paths, wrong class count, etc.) before training kicks off.
!cat dataset/data.yaml

path: /content/dataset
train: train/images
val: val/images
test: test/images

nc: 1
names: ['person']


---

## 2. Training configuration

Settings tuned for a small, on-distribution dataset (~300-500 frames).
Key choices and why:

- **epochs=100**: enough for the model to converge on a small dataset
  without overfitting catastrophically. We'll monitor val loss and can
  stop early if it diverges.
- **imgsz=640**: standard YOLOv8 input size. Bigger means slower training
  and inference but better small-object detection. 640 is the sweet spot
  for our use case.
- **batch=16**: fits comfortably in T4 GPU memory. Lower if you hit OOM.
- **patience=20**: early stopping if val loss hasn't improved in 20
  epochs. Saves time when the model has already converged.
- **augment=True**: YOLOv8's built-in augmentation -- mosaic, mixup,
  HSV shifts, flips. Compensates for low person-diversity in our dataset.

In [ ]:
# Training hyperparameters. All in one place so future tweaks are easy.
TRAINING_CONFIG = {
    "data": "/content/dataset/data.yaml",
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "patience": 20,
    "project": "/content/drive/MyDrive/APCOMS_YOLO_runs",
    "name": "shuttle_finetune_v1",
    "exist_ok": True,
    "augment": True,
    "verbose": True,
}

# Sanity check: print the config so we can confirm before kicking off
# a training run that might take 15-30 minutes.
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

  data: /content/dataset/data.yaml
  epochs: 100
  imgsz: 640
  batch: 16
  patience: 20
  project: /content/drive/MyDrive/APCOMS_YOLO_runs
  name: shuttle_finetune_v1
  exist_ok: True
  augment: True
  verbose: True


In [ ]:
# Force-reinstall numpy to a version that matches what was compiled
# against everything else in the Colab environment. After this we
# need to restart the runtime so the new numpy actually loads.
!pip install -q --force-reinstall numpy==2.0.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultralytics 8.3.0 requires numpy<2.0.0,>=1.23.0, but you have numpy 2.0.2 which is incompatible.


In [ ]:
# Nuke wandb entirely -- we don't need cloud experiment tracking,
# and its overzealous auto-integration with ultralytics keeps
# rejecting our save path. Killing it removes the dependency.
!pip uninstall -y wandb

---

## 3. Train the model

Loads base YOLOv8n weights (pre-trained on COCO) and continues
training on our shuttle dataset. This is "transfer learning" -- we
keep COCO's general feature extractors and just adjust the final
layers to specialize for shuttle entrance detection.

Expected runtime: 15-30 minutes on a T4 GPU. Live metrics print
after each epoch. Best weights are saved automatically to
APCOMS_YOLO_runs/shuttle_finetune_v1/weights/best.pt

In [ ]:
from ultralytics import YOLO

# Load base COCO weights as our starting point. Ultralytics downloads
# the file automatically the first time it's referenced.
model = YOLO("yolov8n.pt")

# Kick off training. The unpacked TRAINING_CONFIG dict provides every
# argument YOLO needs -- keeps the call signature clean.
results = model.train(**TRAINING_CONFIG)

New https://pypi.org/project/ultralytics/8.4.67 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/content/dataset/data.yaml, epochs=100, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=/content/drive/MyDrive/APCOMS_YOLO_runs, name=shuttle_finetune_v1, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=True, agnostic_nms=False, classes=None, retina_ma

train: Scanning /content/dataset/train/labels... 228 images, 5 backgrounds, 0 corrupt: 100%|██████████| 228/228 [00:00<00:00, 2034.69it/s]

train: New cache created: /content/dataset/train/labels.cache


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))


/usr/local/lib/python3.12/dist-packages/ultralytics/data/augment.py:1850: UserWarning: Argument(s) 'quality_lower' are not valid for transform ImageCompression
  A.ImageCompression(quality_lower=75, p=0.0),
val: Scanning /content/dataset/val/labels... 65 images, 0 backgrounds, 0 corrupt: 100%|██████████| 65/65 [00:00<00:00, 1486.09it/s]

...

     93/100      2.31G     0.3927     0.3539     0.8845          5        640: 100%|██████████| 15/15 [00:05<00:00,  2.74it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:00<00:00,  3.66it/s]

                   all         65         80      0.915      0.939      0.966      0.778
EarlyStopping: Training stopped early as no improvement observed in last 20 epochs. Best results observed at epoch 73, best model saved as best.pt.
To update EarlyStopping(patience=20) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



93 epochs completed in 0.188 hours.
Optimizer stripped from /content/drive/MyDrive/APCOMS_YOLO_runs/shuttle_finetune_v1/weights/last.pt, 5.6MB
Optimizer stripped from /content/drive/MyDrive/APCOMS_YOLO_runs/shuttle_finetune_v1/weights/best.pt, 5.6MB

Validating /content/drive/MyDrive/APCOMS_YOLO_runs/shuttle_finetune_v1/weights/best.pt...
Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients, 6.8 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:17<00:00,  5.73s/it]


                   all         65         80      0.924      0.925      0.953      0.808
Speed: 0.2ms preprocess, 259.5ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/drive/MyDrive/APCOMS_YOLO_runs/shuttle_finetune_v1


---

## 4. Validate the fine-tuned model

Evaluates our trained model on the held-out test set -- frames the
model never saw during training. This gives us an honest accuracy
number.

Metrics produced:
- **mAP@50**: mean Average Precision at IoU threshold 0.5.
  YOLOv8's headline accuracy number -- higher is better.
- **mAP@50-95**: stricter version (averaged across IoU thresholds
  0.5 to 0.95). The standard YOLO benchmark metric.
- **Precision**: of all detections, what fraction were correct.
- **Recall**: of all real people, what fraction did we detect.

For shuttle counting, recall matters slightly more than precision --
missing a passenger is worse than briefly seeing a phantom one
(which the tracker filters out anyway).

In [ ]:
# Load the freshly-trained best.pt from this run.
best_weights_path = (
    "/content/drive/MyDrive/APCOMS_YOLO_runs/"
    "shuttle_finetune_v1/weights/best.pt"
)
finetuned_model = YOLO(best_weights_path)

# Run validation on the test split. data.yaml already points YOLO
# at the correct test images and labels.
val_results = finetuned_model.val(
    data="/content/dataset/data.yaml",
    split="test",
    imgsz=640,
    verbose=True,
)

print("\nFine-tuned model metrics on test set:")
print(f"  mAP@50:     {val_results.box.map50:.4f}")
print(f"  mAP@50-95:  {val_results.box.map:.4f}")
print(f"  Precision:  {val_results.box.mp:.4f}")
print(f"  Recall:     {val_results.box.mr:.4f}")

Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 186 layers, 2,684,563 parameters, 0 gradients, 6.8 GFLOPs


val: Scanning /content/dataset/test/labels... 34 images, 0 backgrounds, 0 corrupt: 100%|██████████| 34/34 [00:00<00:00, 1490.70it/s]

val: New cache created: /content/dataset/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.24it/s]


                   all         34         43          1      0.901      0.967      0.716
Speed: 0.4ms preprocess, 26.4ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to runs/detect/val

Fine-tuned model metrics on test set:
  mAP@50:     0.9669
  mAP@50-95:  0.7155
  Precision:  1.0000
  Recall:     0.9014


---

## 5. Compare against the base COCO model

The whole point of fine-tuning is producing a model BETTER than the
generic base. Here we run the base YOLOv8n (untouched, COCO-trained)
on the exact same test set and compare metrics.

This side-by-side comparison is the headline result for the project
report and panel presentation. Even modest improvements
(say, +5% recall) are meaningful evidence that domain-specific
fine-tuning pays off.

In [ ]:
# Load the untouched base model -- exactly what counting_module ships
# with today. Ultralytics caches the download from Section 3 so this
# is instant.
base_model = YOLO("yolov8n.pt")

# Identical validation call, just on the base model. Same test images,
# same image size, same metric calculation -- only the weights differ.
base_results = base_model.val(
    data="/content/dataset/data.yaml",
    split="test",
    imgsz=640,
    verbose=True,
)

print("\nBase COCO model metrics on test set:")
print(f"  mAP@50:     {base_results.box.map50:.4f}")
print(f"  mAP@50-95:  {base_results.box.map:.4f}")
print(f"  Precision:  {base_results.box.mp:.4f}")
print(f"  Recall:     {base_results.box.mr:.4f}")

Ultralytics 8.3.0 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 168 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs


val: Scanning /content/dataset/test/labels.cache... 34 images, 0 backgrounds, 0 corrupt: 100%|██████████| 34/34 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  2.40it/s]


                   all         34         43      0.746      0.744      0.774      0.567
                person         34         43      0.746      0.744      0.774      0.567
Speed: 0.4ms preprocess, 9.4ms inference, 0.0ms loss, 2.7ms postprocess per image
Results saved to runs/detect/val2

Base COCO model metrics on test set:
  mAP@50:     0.7737
  mAP@50-95:  0.5669
  Precision:  0.7465
  Recall:     0.7442


In [ ]:
# Side-by-side comparison table. Prints the absolute numbers AND
# the deltas so we can see at a glance how much fine-tuning helped.
def pct_delta(new, old):
    if old == 0:
        return "n/a"
    return f"{(new - old) * 100:+.2f}pp"

print(f"\n{'Metric':<14}{'Base COCO':>12}{'Fine-tuned':>14}{'Delta':>12}")
print("-" * 52)
print(
    f"{'mAP@50':<14}"
    f"{base_results.box.map50:>12.4f}"
    f"{val_results.box.map50:>14.4f}"
    f"{pct_delta(val_results.box.map50, base_results.box.map50):>12}"
)
print(
    f"{'mAP@50-95':<14}"
    f"{base_results.box.map:>12.4f}"
    f"{val_results.box.map:>14.4f}"
    f"{pct_delta(val_results.box.map, base_results.box.map):>12}"
)
print(
    f"{'Precision':<14}"
    f"{base_results.box.mp:>12.4f}"
    f"{val_results.box.mp:>14.4f}"
    f"{pct_delta(val_results.box.mp, base_results.box.mp):>12}"
)
print(
    f"{'Recall':<14}"
    f"{base_results.box.mr:>12.4f}"
    f"{val_results.box.mr:>14.4f}"
    f"{pct_delta(val_results.box.mr, base_results.box.mr):>12}"
)


Metric           Base COCO    Fine-tuned       Delta
----------------------------------------------------
mAP@50              0.7737        0.9669    +19.31pp
mAP@50-95           0.5669        0.7155    +14.86pp
Precision           0.7465        1.0000    +25.35pp
Recall              0.7442        0.9014    +15.72pp


---

## 6. Export final weights

The best-performing weights from training live in Drive at:
`APCOMS_YOLO_runs/shuttle_finetune_v1/weights/best.pt`

To deploy: download that file and replace `counting_module/models/yolov8n.pt`
in the APCOMS repo. The counting pipeline will automatically pick up
the new weights on its next run -- no code changes needed because
ObjectDetection reads from the same path.

For panel demos, keep BOTH versions: the original COCO weights and
the fine-tuned weights, so you can show before/after detection
quality if questioned.

In [ ]:
# Confirm the trained weights actually landed in Drive and print
# the size as a sanity check. Expect ~6 MB for YOLOv8n.
import os

weights_path = (
    "/content/drive/MyDrive/APCOMS_YOLO_runs/"
    "shuttle_finetune_v1/weights/best.pt"
)

if os.path.exists(weights_path):
    size_mb = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"Fine-tuned weights ready at:")
    print(f"  {weights_path}")
    print(f"  Size: {size_mb:.2f} MB")
    print("\nDownload this file from Drive and place it at:")
    print("  counting_module/models/yolov8n.pt")
else:
    print(f"WARNING: weights not found at {weights_path}")
    print("Check that Section 3 (training) completed successfully.")

Fine-tuned weights ready at:
  /content/drive/MyDrive/APCOMS_YOLO_runs/shuttle_finetune_v1/weights/best.pt
  Size: 5.34 MB

Download this file from Drive and place it at:
  counting_module/models/yolov8n.pt


In [ ]:
# Optional: list everything saved in the training run folder. Useful
# for finding the training plots (results.png, confusion_matrix.png,
# PR curves) that we can include in the project report and panel
# slides as visual evidence of what training looked like.
run_dir = "/content/drive/MyDrive/APCOMS_YOLO_runs/shuttle_finetune_v1"
print(f"Contents of {run_dir}:")
for root, dirs, files in os.walk(run_dir):
    level = root.replace(run_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")

Contents of /content/drive/MyDrive/APCOMS_YOLO_runs/shuttle_finetune_v1:
shuttle_finetune_v1/
  args.yaml
  events.out.tfevents.1781503186.3b13263123bc.6611.0
  events.out.tfevents.1781503319.3b13263123bc.6611.1
  events.out.tfevents.1781503667.3b13263123bc.10036.0
  labels_correlogram.jpg
  labels.jpg
  train_batch0.jpg
  train_batch1.jpg
  train_batch2.jpg
  results.csv
  train_batch1350.jpg
  train_batch1351.jpg
  train_batch1352.jpg
  val_batch0_labels.jpg
  val_batch0_pred.jpg
  val_batch1_labels.jpg
  val_batch1_pred.jpg
  val_batch2_labels.jpg
  val_batch2_pred.jpg
  PR_curve.png
  F1_curve.png
  P_curve.png
  R_curve.png
  confusion_matrix_normalized.png
  confusion_matrix.png
  results.png
  weights/
    last.pt
    best.pt
